# Modellphase
## 1. EBM Modell (inherently interpretable model)

### Schritt 1: Das Problem für EBM übersetzen (Klassifikation statt Ranking)

Da EBM von Natur aus kein "Ranking-Modell" ist (es kann Startlisten nicht direkt vergleichen), müssen wir das Problem für die Phase A in eine binäre Klassifikation (Ja/Nein-Frage) umwandeln.

- Die Frage an das Modell lautet: „Landet dieser spezifische Fahrer auf dieser spezifischen Etappe in den Top 3/10/20?“

- Das Target (y): Aus der Spalte rank machen wir eine neue Zielvariable (z. B. is_top_3). Wenn ein Fahrer Platz 1, 2 oder 3 belegt hat, bekommt er eine 1 (Ja), alle anderen Fahrer (Platz 4 bis 160) bekommen eine 0 (Nein).

### Schritt 2: Die zeitlich korrekte Datentrennung (Chronological Split)

Im Radsport darf man die Daten niemals zufällig mischen (Random Split). Wenn das Modell mit Daten aus der Tour de France 2025 trainiert wird, darf es nicht die Tour de France 2010 vorhersagen – das wäre in der Wissenschaft ein schwerer methodischer Fehler (Data Leakage durch Zeit).

- Trainings-Daten: Wir füttern das Modell mit allen historischen Etappen von 2005 bis einschließlich 2023. Hier lernt das EBM die Muster.

- Test-/Validierungs-Daten: Wir halten die Jahre 2024 und 2025 komplett unter Verschluss. Sie dienen als unsere "Zukunft", an der wir später prüfen, wie gut das Modell wirklich ist.

### Schritt 3: Die Metadaten maskieren

Wenn wir das EBM trainieren, darf es nur mit echten, mathematischen Features lernen. Spalten wie der Klartextname des Fahrers, das Team oder die Etappen-URL würden das Modell verwirren oder zum Absturz bringen.
D.h. wir maskieren alle Spalten mit dem meta_prefix.


### Schritt 4: Das Training der "Glass-Box" (EBM)

Jetzt wird das EBM-Modell auf den Trainingsdaten (2005–2023) gestartet.

- Wie es intern lernt: EBM schaut sich jedes Feature (wie gradient_final_km, rider_bmi, weather_temp_mean) komplett isoliert an und baut für jedes einzelne Feature eine mathematische Kurve (einen sogenannten Shape Plot). 
- Es lernt zum Beispiel: „Wenn die Steigung gegen 0% geht, steigt die Wahrscheinlichkeit für Sprinter. 
- Wenn sie über 7% steigt, stürzt ihre Wahrscheinlichkeit ab und die der Kletterer schießt nach oben.“

- Da es keine unkontrollierten Interaktionen zwischen den Features erlaubt, bleibt dieses gelernte Wissen zu 100% transparent und nachvollziehbar.

### Schritt 5: Die Brücke zum Ranking bauen (Post-Processing)

Wenn das Modell trainiert ist, jagen wir unsere Testdaten (die Jahre 2024 & 2025) durch das EBM.

- Das EBM schaut sich die Startliste von beispielsweise Etappe 14 der Tour de France 2024 an.

- Für jeden einzelnen Fahrer auf dieser Startliste berechnet das EBM eine exakte Gewinn-Wahrscheinlichkeit zwischen 0% und 100%.

- Der Ranking-Trick: Wir nehmen diese berechneten Wahrscheinlichkeiten und sortieren die Startliste dieser Etappe in Pandas einfach abwärts (höchste Wahrscheinlichkeit zuerst).

- Das Ergebnis: Der Fahrer mit der höchsten EBM-Wahrscheinlichkeit wird unser prognostizierter Platz 1, der zweite Platz 2, usw. Damit haben wir aus einem Klassifikationsmodell ein fertiges Etappen-Ranking gebaut!

### Schritt 6: Die primäre Erklärung herbeiführen

Zum Schluss nutzen wir die integrierte Visualisierung der EBM-Bibliothek (interpret). Wir lassen uns die gelernten Kurven (Shape Plots) der wichtigsten Features ausgeben – allen voran für euer brandneues Feature gradient_final_km.


In [67]:
import os
import pandas as pd
import numpy as np
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from interpret import show
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, precision_score, recall_score
import itertools
import time
import pickle

print(pd.__version__)


2.3.3


In [20]:
pickle_path = '../../data/processed/25_cleaned_master_data.pkl'

df = pd.read_pickle(pickle_path)

### Schritt 1 Schritt 1: Das Problem für EBM übersetzen (Klassifikation statt Ranking)

In [21]:
df['rank'].dtype
df['rank'].astype(int)

0           1
1           2
2           3
3           4
4           5
         ... 
196043    149
196044    150
196045    151
196046    152
196047    153
Name: rank, Length: 196048, dtype: int64

In [22]:
for col in df.columns:
    print(f"{col}: {df[col].dtype}")

meta_race: category
meta_year: int64
meta_url: object
rank: float64
meta_rider_url: object
height: float64
meta_name: object
meta_nationality: category
weight: int64
meta_url_name: object
meta_departure: object
meta_arrival: object
distance: float64
vertical_meters: int64
one_day_races: int64
gc: int64
time_trial: int64
sprint: int64
climber: int64
hills: int64
stage_nr: int64
meta_date: datetime64[ns]
meta_departure_lat: float64
meta_departure_lon: float64
meta_arrival_lat: float64
meta_arrival_lon: float64
rider_points_season: int64
rider_rank_season: int64
meta_current_team: object
team_tier: category
age_at_race: int64
rider_bmi: float64
race_competitiveness_median: float64
team_power_index: float64
wind_stability_index: float64
weather_temp_mean: float64
weather_temp_trend: float64
weather_rain_prob_mean: float64
weather_precipitation_mean: float64
weather_humidity_mean: float64
won_how_cat: category
gradient_final_km: float64


In [23]:
# Wir erstellen das Top-5/10 Target (1 = Top5/10/20, 0 = Plätze 6+/10+/20+)
df['target_top_5'] = np.where(df['rank'] <= 5, 1, 0)

df['target_top_10'] = np.where(df['rank'] <= 10, 1, 0)

df['target_top_20'] = np.where(df['rank'] <= 20, 1, 0)

In [24]:
print(df['target_top_5'].value_counts().to_string())
print(df['target_top_10'].value_counts().to_string())
print(df['target_top_20'].value_counts().to_string())


# Verteilung der 1 und 0 Werte

target_top_5
0    190197
1      5851
target_top_10
0    184368
1     11680
target_top_20
0    172740
1     23308


### Schritt 2: Die zeitlich korrekte Datentrennung (Chronological Split) + Schritt 3: Die Metadaten maskieren

In [25]:
chosen_target = 'target_top_10'
# Für ersten Lauf top 10 nutzen

# Automatische Feature-Selektion:
# Wir filtern alle Spalten heraus, die mit 'meta_' beginnen oder eines der Targets sind.
features = [col for col in df.columns if not col.startswith('meta_')
            and col not in ['rank', 'rank_numeric', 'target_top_5', 'target_top_10', 'target_top_20'] and 'won_how_cat' not in col]

print(f"Selektierte Lern-Features {len(features)} Stück für das EBM-Modell:")
print()

# --- TRAININGS-DATEN (Historie: 2005 bis einschließlich 2023) ---
df_train_historical = df[df['meta_year'] <= 2023]
X_train = df_train_historical[features]
y_train = df_train_historical[chosen_target]

# --- TEST-DATEN (Zukunft: Saisons 2024 und 2025) ---
df_test_future = df[df['meta_year'] >= 2024]
X_test = df_test_future[features]
y_test = df_test_future[chosen_target]

print("ERGEBNIS DES CHRONOLOGISCHEN SPLITS:")
print(f"[TRAIN] X_train (Features 2005-2023): {X_train.shape[0]} Zeilen x {X_train.shape[1]} Spalten")
print(f"[TRAIN] y_train (Labels   2005-2023): {y_train.shape[0]} Zeilen")
print(f"[TEST] X_test  (Features 2024-2025): {X_test.shape[0]} Zeilen x {X_test.shape[1]} Spalten")
print(f"[TEST] y_test  (Labels   2024-2025): {y_test.shape[0]} Zeilen")


Selektierte Lern-Features 25 Stück für das EBM-Modell:

ERGEBNIS DES CHRONOLOGISCHEN SPLITS:
[TRAIN] X_train (Features 2005-2023): 178246 Zeilen x 25 Spalten
[TRAIN] y_train (Labels   2005-2023): 178246 Zeilen
[TEST] X_test  (Features 2024-2025): 17802 Zeilen x 25 Spalten
[TEST] y_test  (Labels   2024-2025): 17802 Zeilen


### Schritt 4: Das Training der "Glass-Box" (EBM)

In [26]:
# 1. Wir berechnen die Gewichte händisch mit Scikit-Learn, um das Klassenungleichgewicht auszugleichen
# Da nur ca. 6-7% in den Top 10 landen, bekommen die '1er' ein höheres Gewicht als die '0er'
sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

In [2]:

# 2. Initialisieren des EBM
ebm = ExplainableBoostingClassifier(
    interactions=0,
    validation_size= 0.15,
    outer_bags=1,
    max_rounds=5000,
    learning_rate=0.015,
    random_state=42,
    n_jobs=1 # Nutzt alle Prozessorkerne deines PCs für maximale Geschwindigkeit
)

# 3. Wir übergeben die berechneten Gewichte direkt beim mathematischen Training
ebm.fit(X_train, y_train, sample_weight=sample_weights_train)

NameError: name 'ExplainableBoostingClassifier' is not defined

In [1]:
show(ebm.explain_global())

NameError: name 'show' is not defined

- die Attribute der Fahrer, die von PCS kommen (climber, sprint, gc), sowie die Ranglistenpositionen sind sehr stark

### Schritt 5: Die Brücke zum Ranking bauen (Post-Processing)

In [29]:
# Wir berechnen die Wahrscheinlichkeiten (Wahrscheinlichkeit für Klasse 1, also Top 10)
# predict_proba gibt [Wahrscheinlichkeit für 0, Wahrscheinlichkeit für 1] aus, daher [:, 1]
y_pred_proba = ebm.predict_proba(X_test)[:, 1]

# Wir machen auch eine harte Ja/Nein-Vorhersage (0 oder 1) basierend auf dem Standard-Threshold
y_pred_binary = ebm.predict(X_test)

# Wir hängen die Vorhersagen an das originale Test-DataFrame mit den Metadaten an!
# Dadurch verknüpfen wir die mathematische Vorhersage wieder mit Fahrernamen und Etappen.
df_results = df_test_future.copy()
df_results['ebm_prob'] = y_pred_proba
df_results['ebm_pred_binary'] = y_pred_binary

In [30]:
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score auf den Testdaten (2024-2025): {auc_score:.4f}")

ROC-AUC Score auf den Testdaten (2024-2025): 0.8683


In [31]:
# 2. Classification Report (Precision, Recall, F1-Score)
print("Detaillierter Klassifikations-Report:")
print(classification_report(y_test, y_pred_binary))

Detaillierter Klassifikations-Report:
              precision    recall  f1-score   support

           0       0.98      0.75      0.85     16698
           1       0.18      0.82      0.30      1104

    accuracy                           0.76     17802
   macro avg       0.58      0.79      0.57     17802
weighted avg       0.93      0.76      0.82     17802



In [32]:
print("Konfusionsmatrix")
print(confusion_matrix(y_test, y_pred_binary))

Konfusionsmatrix
[[12586  4112]
 [  198   906]]


### 4 + 5 ohne die Features aus PCS (Onedayraces, GC, TT, Sprint, Climer, Hills)
- aus PCS (Onedayraces, GC, TT, Sprint, Climer, Hills)
    - Werte sind Features aus der PCS Website
    - kaum nachzuvollziehen wie diese zu stande kommen
    - somit nehmen wir diese erstmal raus.
- Height und Weight, da wir die Spalte BMI haben
- zusätzlich wollen wir die rider_points per Season nicht mit betrachten, da diese ja immer für die gesamte Saison gelten und das Modell somit Schlüsse auf die Performance bei den großen Rundfahrten ziehen kann
- dies betrifft auch die Teamstärke sowie die Rennstärke, da diese alle auf der Ranglistenposition basieren.

In [33]:
for col in df.columns:
    print(f"{col}: {df[col].dtype}")

meta_race: category
meta_year: int64
meta_url: object
rank: float64
meta_rider_url: object
height: float64
meta_name: object
meta_nationality: category
weight: int64
meta_url_name: object
meta_departure: object
meta_arrival: object
distance: float64
vertical_meters: int64
one_day_races: int64
gc: int64
time_trial: int64
sprint: int64
climber: int64
hills: int64
stage_nr: int64
meta_date: datetime64[ns]
meta_departure_lat: float64
meta_departure_lon: float64
meta_arrival_lat: float64
meta_arrival_lon: float64
rider_points_season: int64
rider_rank_season: int64
meta_current_team: object
team_tier: category
age_at_race: int64
rider_bmi: float64
race_competitiveness_median: float64
team_power_index: float64
wind_stability_index: float64
weather_temp_mean: float64
weather_temp_trend: float64
weather_rain_prob_mean: float64
weather_precipitation_mean: float64
weather_humidity_mean: float64
won_how_cat: category
gradient_final_km: float64
target_top_5: int64
target_top_10: int64
target_top_20

In [49]:
intransparente_features = [
    'climber', 'sprint', 'hills', 'gc', 'time_trial', 'one_day_races', 'height', 'weight',
    'rider_points_season', 'rider_rank_season', 'race_competitiveness_median', 'team_power_index'
]

features_b = [col for col in df.columns if not col.startswith('meta_')
                 and 'target' not in col
                 and 'rank' not in col
                 and 'won_how_cat' not in col
                 and col not in intransparente_features]


X_train_b = df_train_historical[features_b]
X_test_b = df_test_future[features_b]


In [50]:
print("Anzahl Features:", len(X_train_b.columns))
print("--- Spaltenliste ---")
for i, col in enumerate(X_train_b.columns, 1):
    print(f"{i:02d}. {col} ({X_train_b[col].dtype})")

Anzahl Features: 13
--- Spaltenliste ---
01. distance (float64)
02. vertical_meters (int64)
03. stage_nr (int64)
04. team_tier (category)
05. age_at_race (int64)
06. rider_bmi (float64)
07. wind_stability_index (float64)
08. weather_temp_mean (float64)
09. weather_temp_trend (float64)
10. weather_rain_prob_mean (float64)
11. weather_precipitation_mean (float64)
12. weather_humidity_mean (float64)
13. gradient_final_km (float64)


In [51]:
ebm_b = ExplainableBoostingClassifier(
    interactions=0,
    validation_size= 0.15,
    outer_bags=1,
    max_rounds=5000,
    learning_rate=0.015,
    random_state=42,
    n_jobs=1
)

ebm_b.fit(X_train_b, y_train, sample_weight=sample_weights_train)

,feature_names,None
,feature_types,None
,max_bins,1024
,max_interaction_bins,64
,interactions,0
,exclude,None
,validation_size,0.15
,outer_bags,1
,inner_bags,0
,learning_rate,0.015
,greedy_ratio,10.0


In [ ]:
show(ebm_b.explain_global())

In [53]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# 1. Wahrscheinlichkeiten für die Testdaten (Modell B) berechnen
y_pred_proba_b = ebm_b.predict_proba(X_test_b)[:, 1]

# 2. Harte Ja/Nein-Vorhersagen (0 oder 1) für den Classification Report berechnen
y_pred_binary_b = ebm_b.predict(X_test_b)

# 3. ROC-AUC Score berechnen und ausgeben
auc_score_b = roc_auc_score(y_test, y_pred_proba_b)
print(f"ROC-AUC Score (Modell B): {auc_score_b:.4f}")
print()

# 4. Detaillierter Classification Report (Precision, Recall, F1-Score)
print("Detaillierter Klassifikations-Report für Modell B:")
print(classification_report(y_test, y_pred_binary_b))
print()

# 5. Confusion Matrix ausgeben
print("Konfusionsmatrix für Modell B:")
print(confusion_matrix(y_test, y_pred_binary_b))


ROC-AUC Score (Modell B): 0.5817

Detaillierter Klassifikations-Report für Modell B:
              precision    recall  f1-score   support

           0       0.95      0.67      0.78     16698
           1       0.08      0.46      0.14      1104

    accuracy                           0.65     17802
   macro avg       0.52      0.56      0.46     17802
weighted avg       0.90      0.65      0.74     17802


Konfusionsmatrix für Modell B:
[[11105  5593]
 [  599   505]]


Somit haben wir ein Modell, in dem wir Stur alle verbleibenden Features hineingegeben haben und eines bei dem wir Data Leakage Features entfernt haben.

Der Rückgang der Performance ist nicht zu übersehen.


Im nächsten Schritt betrachten wir die Hyperparameter und analysieren, welche sich für ein Tuning und somit für ein Modellbenchmarking eignen.

### Hyperparameterbetrachtung

### 1. Konstante Parameter (Standard-Datenhandling und Stabilität)

Diese Parameter werden festgeschrieben. Sie sichern die mathematische Korrektheit, das Datenhandling sowie die Stabilität des Trainingsprozesses und bleiben über alle Durchläufe hinweg unverändert.

* **feature_names=None, feature_types=None, exclude=None, callback=None, monotone_constraints=None:** Standardeinstellungen für das interne Datenhandling des Algorithmus.
* **objective='log_loss':** Mathematische Zielfunktion (Logarithmischer Verlust) für die binäre Klassifikation zur Trennung der beiden Klassen (0 oder 1).
* **missing='separate':** Behandlung fehlender Werte (insbesondere bei meteorologischen Variablen) als eigenständige Kategorie bzw. separater Split, um Informationsverlust zu verhindern.
* **validation_size=0.15, early_stopping_rounds=100, early_stopping_tolerance=1e-05:** Konfiguration des vorzeitigen Abbruchs (Early Stopping). Verbessert sich das Modell über 100 Runden hinweg auf den 15 % Validierungsdaten nicht um den tolerierten Mindestwert, bricht das Training ab. Dies dient als primärer Schutz vor Overfitting.
* **random_state=42:** Festlegung des Zufallssatkes zur Gewährleistung der vollständigen Reproduzierbarkeit aller Testergebnisse bei wiederholten Durchläufen.
* **n_jobs=1:** Begrenzung auf einen einzigen CPU-Thread, um Ressourcenkonflikte, Deadlocks und das Einfrieren des Python-Kernels in der Ausführungsumgebung zu verhindern.
* **min_samples_leaf=4, min_hessian=0.0001, min_cat_samples=10, cat_smooth=10.0, max_leaves=2:** Standardmäßige strukturelle Restriktionen für die zugrundeliegenden Entscheidungsbäume des EBM. Da ein additives Modell trainiert wird, besitzen diese Parameter eine untergeordnete Sensitivität.

---

### 2. Rechenzeit-Parameter (Konstant limitierte Ressourcenwächter)

Diese Parameter beeinflussen maßgeblich die Rechenleistung und Systemauslastung, bieten jedoch ab einem gewissen Schwellenwert keinen signifikanten informationellen Mehrwert mehr. Sie werden konstant auf effizienten Werten gehalten.

* **max_bins=1024:** Bestimmt die maximale Anzahl der Intervalle (Bins), in die kontinuierliche Variablen (z. B. Rider BMI, Temperatur) diskretisiert werden. Der Wert 1024 sichert eine hohe Granularität bei moderater Speicherlast.
* **max_rounds=5000:** Die absolute Obergrenze der Boosting-Iterationen. Aufgrund des aktiven Early Stoppings dient dieser Wert lediglich als technischer Sicherheitsdeckel.
* **max_interaction_bins=64, interaction_smoothing_rounds=75:** Spezifische Parameter für die Glättung und Quantisierung von Wechselwirkungen. Sie sind bei Konfigurationen ohne Interaktionen funktionslos.

---

### 3. Tuning-Parameter (Variablen für die experimentelle Matrix)

Diese Hyperparameter werden im Rahmen eines systematischen Grid-Search variiert, um die optimale Balance zwischen Modellkomplexität, Generalisierungsfähigkeit und prädiktiver Güte zu ermitteln.

#### Variable 1: interactions (Modellkomplexität)
* **Setup A: 0** (Reines, ungestörtes 1D-Basismodell zur Erfassung isolierter Haupteffekte).
* **Setup B: 3** (Erhöhung auf exakt drei paarweise Interaktionen zur Abbildung kombinatorischer Effekte).
* *Begründung:* Der Vergleich zeigt empirisch, ob die explizite Modellierung von nicht-linearen Wechselwirkungen (z. B. das Zusammenspiel aus topographischer Steigung und biometrischem BMI) die Metriken gegenüber dem reinen Haupteffekt-Modell steigert.

#### Variable 2: learning_rate (Schrittweite des Gradientenverfahrens)
* **Setup A: 0.005** (Extrem konservatives, feingranulares Lernen).
* **Setup C: 0.050** (Beschleunigte Konvergenz).
* *Begründung:* Die Skalierung der Lernrate untersucht die mathematische Balance zwischen zu schnellem Lernen (Risiko des Overfittings) und zu langsamem Lernen (Risiko der Unteranpassung innerhalb der maximalen Runden).

#### Variable 3: outer_bags (Varianzreduktion durch Ensembling)
* **Setup A: 1** (Singulärer Modell-Durchlauf zur schnellen Approximation).
Wir belassen es beim Benchmarking bei einem Outerbag, um die Rechenzeit nicht in die Höhe zu treiben.
Wenn wir unsere Output Modell haben, können wir für die besten x Modelle den Outer Bag auf 3 erhöhen.

In [63]:
# 1. zu betrachtenden Features
features_benchmark = [
    'distance', 'vertical_meters', 'stage_nr', 'team_tier', 'age_at_race',
    'rider_bmi', 'wind_stability_index', 'weather_temp_mean', 'weather_temp_trend',
    'weather_rain_prob_mean', 'weather_precipitation_mean', 'weather_humidity_mean',
    'gradient_final_km'
]

# 2. Historischen Train- und Zukunfts-Test-Split erstellen
df_train_historical = df[df['meta_year'] <= 2023]
df_test_future = df[df['meta_year'] >= 2024]

X_train = df_train_historical[features_benchmark]
X_test = df_test_future[features_benchmark]

# 3. Die Targets (Zielvariablen) festlegen
targets = ['target_top_5', 'target_top_10', 'target_top_20']

# 4. Die Tuning-Parameter als Listen definieren (Eure wissenschaftliche Matrix)
grid_interactions = [0, 3]
grid_learning_rates = [0.005, 0.050]
grid_outer_bags = [1]

# 5. Alle mathematischen Kombinationen der Parameter generieren (Kartesisches Produkt)
parameter_combinations = list(itertools.product(grid_interactions, grid_learning_rates, grid_outer_bags))

print(f"Daten erfolgreich vorbereitet.")
print(f"Anzahl Zeilen im Trainings-Set (<=2023): {X_train.shape[0]}")
print(f"Anzahl Zeilen im Test-Set (>=2024): {X_test.shape[0]}")
print(f"Generierte Parameter-Kombinationen pro Target: {len(parameter_combinations)}")
print(f"Gesamtzahl der Modell-Runs (3 Targets * {len(parameter_combinations)} Kombis): {3 * len(parameter_combinations)}")

Daten erfolgreich vorbereitet.
Anzahl Zeilen im Trainings-Set (<=2023): 178246
Anzahl Zeilen im Test-Set (>=2024): 17802
Generierte Parameter-Kombinationen pro Target: 4
Gesamtzahl der Modell-Runs (3 Targets * 4 Kombis): 12


Somit kommen wir auf 12 Modelle, die wir danach Benchmarken wollen  

In [64]:
grid_search_results = []

# Startzeit für die Gesamtmessung
start_total = time.time()

# 1. Äußere Schleife: Die 3 sportlichen Targets
for target_idx, target_col in enumerate(targets, 1):
    print(f"\n[TARGET {target_idx}/3]: {target_col.upper()}")
    print("------------------------------------------------------------------------------------------")

    y_train = df_train_historical[target_col]
    y_test = df_test_future[target_col]

    # Class Imbalance für dieses spezifische Target ausgleichen
    sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

    # 2. Innere Schleife: Die 12 Hyperparameter-Kombinationen
    for combo_idx, (inter, lr, bags) in enumerate(parameter_combinations, 1):
        run_name = f"Combo {combo_idx:02d} (Inter={inter}, LR={lr}, Bags={bags})"
        print(f"Starte {run_name}...", end="")

        start_run = time.time()

        # EBM mit den exakten Parametern eurer Matrix initialisieren
        ebm = ExplainableBoostingClassifier(
            interactions=inter,
            learning_rate=lr,
            outer_bags=bags,
            validation_size=0.15,
            max_rounds=5000,
            early_stopping_rounds=100,
            early_stopping_tolerance=1e-05,
            objective='log_loss',
            missing='separate',
            max_bins=1024,
            random_state=42,
            n_jobs=1
        )

        # Modell trainieren
        ebm.fit(X_train, y_train, sample_weight=sample_weights_train)

        # Vorhersagen für die Metriken generieren
        preds_proba = ebm.predict_proba(X_test)[:, 1]
        preds_binary = ebm.predict(X_test)

        # Metriken berechnen
        auc = roc_auc_score(y_test, preds_proba)
        precision = precision_score(y_test, preds_binary, zero_division=0)
        recall = recall_score(y_test, preds_binary, zero_division=0)

        end_run = time.time()
        duration = end_run - start_run

        print(f" -> Done! AUC: {auc:.4f} | Zeit: {duration:.1f}s")

        # Ergebnisse in die temporäre Liste sichern
        grid_search_results.append({
            "Target": target_col,
            "Interactions": inter,
            "Learning_Rate": lr,
            "Outer_Bags": bags,
            "ROC_AUC": round(auc, 4),
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "Duration_Seconds": round(duration, 1)
        })

end_total = time.time()
print("\n==========================================================================================")
print(f"GRID-SEARCH beendet. Gesamtdauer: {((end_total - start_total)/60):.2f} Minuten")
print("==========================================================================================")

# 3. Ergebnisse in ein wunderschönes DataFrame gießen
df_grid_report = pd.DataFrame(grid_search_results)
display(df_grid_report)


[TARGET 1/3]: TARGET_TOP_5
------------------------------------------------------------------------------------------
Starte Combo 01 (Inter=0, LR=0.005, Bags=1)... -> Done! AUC: 0.6035 | Zeit: 123.1s
Starte Combo 02 (Inter=0, LR=0.05, Bags=1)... -> Done! AUC: 0.6369 | Zeit: 195.5s
Starte Combo 03 (Inter=3, LR=0.005, Bags=1)... -> Done! AUC: 0.6711 | Zeit: 116.2s
Starte Combo 04 (Inter=3, LR=0.05, Bags=1)... -> Done! AUC: 0.6907 | Zeit: 144.9s

[TARGET 2/3]: TARGET_TOP_10
------------------------------------------------------------------------------------------
Starte Combo 01 (Inter=0, LR=0.005, Bags=1)... -> Done! AUC: 0.5687 | Zeit: 210.3s
Starte Combo 02 (Inter=0, LR=0.05, Bags=1)... -> Done! AUC: 0.5960 | Zeit: 177.7s
Starte Combo 03 (Inter=3, LR=0.005, Bags=1)... -> Done! AUC: 0.6645 | Zeit: 163.4s
Starte Combo 04 (Inter=3, LR=0.05, Bags=1)... -> Done! AUC: 0.6772 | Zeit: 123.3s

[TARGET 3/3]: TARGET_TOP_20
------------------------------------------------------------------------

,Target,Interactions,Learning_Rate,Outer_Bags,ROC_AUC,Precision,Recall,Duration_Seconds
0,target_top_5,0,0.005,1,0.6035,0.0417,0.4622,123.1
1,target_top_5,0,0.050,1,0.6369,0.0494,0.4604,195.5
2,target_top_5,3,0.005,1,0.6711,0.0492,0.5935,116.2
3,target_top_5,3,0.050,1,0.6907,0.0542,0.5576,144.9
4,target_top_10,0,0.005,1,0.5687,0.0789,0.4629,210.3
5,target_top_10,0,0.050,1,0.5960,0.0852,0.4484,177.7
6,target_top_10,3,0.005,1,0.6645,0.0932,0.6078,163.4
7,target_top_10,3,0.050,1,0.6772,0.1030,0.5924,123.3
8,target_top_20,0,0.005,1,0.5585,0.1442,0.4857,132.9
9,target_top_20,0,0.050,1,0.5793,0.1550,0.4599,161.1


In [65]:
# Pfad an euren Drive-Ordner anpassen
report_path = '../../data/processed/ebm_benchmark_grid_report.csv'

# Als CSV speichern
df_grid_report.to_csv(report_path, index=False)
print(f"Ergebnistabelle erfolgreich gesichert unter: {report_path}")

Ergebnistabelle erfolgreich gesichert unter: ../../data/processed/ebm_benchmark_grid_report.csv


In [ ]:
# Beispiel: Ihr habt das Champion-Modell für Target Top 5 trainiert (nennen wir es ebm_top_5)
model_save_path = '../../data/processed/ebm_modells.pkl'

# Das Modell als Binärdatei (wb = write binary) auf den Drive schreiben
with open(model_save_path, 'wb') as f:
    pickle.dump(ebm, f)
